# Basic Charts

This notebook answers first-level questions from `studentInfo.csv`, especially around modules/courses and student counts.

Important grain note: `studentInfo.csv` has one row per student-module-presentation attempt. A student can appear more than once if they take more than one module or presentation.

## Current answers from the raw data

- Total student-module-presentation rows: **32,593**
- Unique students: **28,785**
- Students taking more than one module: **2,479**
- Students taking exactly one module: **26,306**
- Students taking exactly two modules: **2,459**
- Students taking exactly three modules: **20**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW_DIR = Path('../data/raw').resolve()
student_info = pd.read_csv(RAW_DIR / 'studentInfo.csv')

student_info.head()

In [ ]:
total_rows = len(student_info)
unique_students = student_info['id_student'].nunique()
modules = sorted(student_info['code_module'].unique())

summary = pd.DataFrame({
    'metric': ['student_module_presentation_rows', 'unique_students', 'number_of_modules'],
    'value': [total_rows, unique_students, len(modules)]
})

summary

## How many students are in each module?

This uses unique students per module. It is different from counting rows because a student may appear in more than one presentation of the same module.

In [ ]:
students_by_module = (
    student_info
    .groupby('code_module')['id_student']
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name='unique_students')
)

students_by_module

In [ ]:
ax = students_by_module.plot(
    kind='bar',
    x='code_module',
    y='unique_students',
    legend=False,
    figsize=(8, 5),
    color='#3b82f6'
)

ax.set_title('Unique Students by Module')
ax.set_xlabel('Module')
ax.set_ylabel('Unique students')
ax.bar_label(ax.containers[0], padding=3)
plt.tight_layout()
plt.show()

## How many rows/attempts are in each module?

This counts student-module-presentation rows. Use this when the analysis is about attempts or presentations rather than unique people.

In [ ]:
attempts_by_module = (
    student_info['code_module']
    .value_counts()
    .rename_axis('code_module')
    .reset_index(name='student_module_presentation_rows')
)

attempts_by_module

In [ ]:
ax = attempts_by_module.plot(
    kind='bar',
    x='code_module',
    y='student_module_presentation_rows',
    legend=False,
    figsize=(8, 5),
    color='#10b981'
)

ax.set_title('Student-Module-Presentation Rows by Module')
ax.set_xlabel('Module')
ax.set_ylabel('Rows')
ax.bar_label(ax.containers[0], padding=3)
plt.tight_layout()
plt.show()

## How many students take more than one module?

In [ ]:
modules_per_student = (
    student_info
    .groupby('id_student')['code_module']
    .nunique()
    .reset_index(name='number_of_modules')
)

students_more_than_one_module = (modules_per_student['number_of_modules'] > 1).sum()
students_more_than_one_module

In [ ]:
students_by_module_count = (
    modules_per_student['number_of_modules']
    .value_counts()
    .sort_index()
    .rename_axis('number_of_modules')
    .reset_index(name='students')
)

students_by_module_count

In [ ]:
ax = students_by_module_count.plot(
    kind='bar',
    x='number_of_modules',
    y='students',
    legend=False,
    figsize=(7, 5),
    color='#f59e0b'
)

ax.set_title('Students by Number of Modules Taken')
ax.set_xlabel('Number of modules')
ax.set_ylabel('Students')
ax.bar_label(ax.containers[0], padding=3)
plt.tight_layout()
plt.show()

## Optional: modules by final result

This chart helps connect module enrollment to outcomes.

In [ ]:
result_by_module = pd.crosstab(
    student_info['code_module'],
    student_info['final_result']
)

result_by_module

In [ ]:
ax = result_by_module.plot(
    kind='bar',
    stacked=True,
    figsize=(9, 5),
    colormap='tab20'
)

ax.set_title('Final Result by Module')
ax.set_xlabel('Module')
ax.set_ylabel('Student-module-presentation rows')
plt.tight_layout()
plt.show()